# DELTA LAKE

Local instance to manage MatrizActividades

In [1]:
import sys
import time
from pathlib import Path
import pandas as pd
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)

DELTA_TABLE_PATH_ON_HOST = "/home/vlad/delta_V30"

table_path = DELTA_TABLE_PATH_ON_HOST



Success!!!


### Conexon con MongoDB

In [5]:
# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


### Conexion con DeltaLake

In [15]:
# DELTA LAKE Connection

# Verify the existence of the DELTA LAKE table
if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {table_path}")



Conectado a la tabla Delta Lake en: /home/vlad/delta_V30


### Analisis de Cambios en una OT

In [3]:
def calcular_minutos_transcurridos( start_times, end_times ):
  """
  This function works because subtracting two pandas Series of datetimes
  is a vectorized operation.
  """
  try:
    # Ensure columns are in datetime format first
    start_times = pd.to_datetime(start_times)
    end_times = pd.to_datetime(end_times)

    time_difference = end_times - start_times
    # Return the difference in minutes
    return (time_difference.dt.total_seconds() / 60).astype(int)
  except Exception as e:
    print(f" EXCEPTION:\n{e}")

In [3]:
#  ANALIZAR UNA OT filtrando por su indice

filtered_df = df.query( f"id_ot == 160106" ).sort_values(by='Item')


In [4]:
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
122948,1,informativa,En la agencia de la EERSSA se coordina los tra...,PROG,·,No,No,No,RUTINARIA,·,...,2025-08-29 07:30:00,2025-08-29 07:40:00,10,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
122949,2,transporte,Traslado desde la agencia de la EERSSA Zamora ...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2025-08-29 07:40:00,2025-08-29 07:50:00,10,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
122950,3,REDES,En el sector de la Fragancia se realiza lo sig...,PROG,Zamora I,No,No,No,CORRECTIVO,·,...,2025-08-29 07:50:00,2025-08-29 12:55:00,305,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
122951,4,lunch,Lunch en La Fragancia.,ALIMEN,·,No,No,No,LUNCH,·,...,2025-08-29 12:55:00,2025-08-29 13:55:00,60,MORALES RIVERA LUIS ALBERTO,2,No,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
122952,5,ALUMBRADO,Sector de la Fragancia se continua con la repa...,PROG,Zamora I,No,No,No,CORRECTIVO,·,...,2025-08-29 13:55:00,2025-08-29 19:29:00,334,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
122953,7,se_labora,SE LABORA: LM y LL de 07:30 a 12:55 y de 13:55...,LABORA,·,No,No,No,·,·,...,2025-08-29 00:00:01,2025-08-29 00:00:02,0,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
122954,8,se_labora,SE LABORA: RY se encuentra con reposo médico.\...,LABORA,·,No,No,No,·,·,...,2025-08-29 00:00:01,2025-08-29 00:00:02,0,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf


In [104]:
#filtered_df.loc[93810, 'InicioEvento'] = '2021-04-23 22:10:00'
filtered_df['Duracion'] = calcular_minutos_transcurridos(
    filtered_df['InicioEvento'],
    filtered_df['FinEvento']
)

### Limpieza de InicioEvento y FinEvento

In [ ]:
# 1. Elimina espacios en blanco del Inicio y del Fin del evento

df['InicioEvento'] = df['InicioEvento'].str.strip()
df['FinEvento'] = df['FinEvento'].str.strip()

In [124]:
# Crear una mascara para los elementos que tienen 20 caracteres
# Create mask - True for rows that DON'T match the pattern
mask = ~df['FinEvento'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$')
# Handle NaN values explicitly
mask = mask | df['FinEvento'].isna()

In [125]:
df[mask].tail(5)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo


In [123]:
# Procesar estas fechas y horas para que todas tengan el mismo formato:

# 2. Funcion para eliminar el componente de Time Zone
#    éste es introducido cuando en actividades no se consigue una 'fecha_moda' 
#    y es necesario utilizar la 'fecha' de hoja_uno.   

def elimina_timezone( fecha ):
  try:
    
    # Se elimina el componente de Zona Horaria
    fecha_inicio = fecha.replace('T', ' ').split()
    fecha_inicio = fecha_inicio[0]+' '+fecha_inicio[-1]
    return fecha_inicio
  
  except:
    print(f"[X] No fue posible convertir la cadena de caracteres:  {fecha}")
    return fecha

df.loc[mask, 'InicioEvento'] = df.loc[mask, 'InicioEvento'].apply( lambda x: elimina_timezone(x))
df.loc[mask, 'FinEvento'] = df.loc[mask, 'FinEvento'].apply( lambda x: elimina_timezone(x))


### Guardar los cambios en DeltaLake

In [17]:
dt = DeltaTable(table_path)
#dt.restore(1979)


In [18]:
dt.version()

2465

### Optimización y Aspirado

In [9]:
dt.optimize.compact()

{'numFilesAdded': 1,
 'numFilesRemoved': 402,
 'filesAdded': '{"avg":15978605.0,"max":15978605,"min":15978605,"totalFiles":1,"totalSize":15978605}',
 'filesRemoved': '{"avg":76402.73631840796,"max":8811865,"min":7966,"totalFiles":402,"totalSize":30713900}',
 'partitionsOptimized': 1,
 'numBatches': 779,
 'totalConsideredFiles': 402,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': True}

In [ ]:
dt.vacuum(retention_hours=2, enforce_retention_duration=False, dry_run=True)


[2025-09-29T19:30:20Z WARN  deltalake_core::kernel::transaction] Attempting to write a transaction 2464 but the underlying table has been updated to 2464
    DefaultLogStore(/home/vlad/delta_V30/)


['part-00001-d57c30d9-10a3-494d-a0a7-16d3282b16af-c000.snappy.parquet',
 'part-00001-a0a15b0d-2812-4135-9fe4-5519df35c7b3-c000.snappy.parquet',
 'part-00001-7dd7a227-d121-493f-b29f-4175e0e384e1-c000.snappy.parquet',
 'part-00001-0f53da6e-5d65-42ce-ac8b-54bc8efc5f5e-c000.snappy.parquet',
 'part-00001-c0d7b295-f7b8-45ce-934c-e78009ca4b45-c000.snappy.parquet',
 'part-00001-7829cf16-b846-4b08-8c3e-6f0a58ea448c-c000.snappy.parquet',
 'part-00001-bf1fb039-dce0-4031-8a66-389a2fcc38ba-c000.snappy.parquet',
 'part-00001-5ee1ed22-402f-4d6a-b449-3ec7e2ec0073-c000.snappy.parquet',
 'part-00001-3e288d50-1c82-463b-89d6-896f026a6ca0-c000.snappy.parquet',
 'part-00001-42699686-c7a4-488e-9690-2e9d5daa5edf-c000.snappy.parquet',
 'part-00001-992c3e68-e768-47a4-8336-b3e1628ea166-c000.snappy.parquet',
 'part-00001-1f140a4a-5108-461c-a755-075b4f84be95-c000.snappy.parquet',
 'part-00001-e6c7f175-e51f-4070-888b-d90b1ad0c15a-c000.snappy.parquet',
 'part-00001-e9af8fb1-8be0-4fd8-9f1f-6aad6e59f61b-c000.snappy.pa

In [106]:
import pyarrow as pa 

edited_data = pa.Table.from_pandas( filtered_df )

try:

    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
                    source=edited_data,
                    predicate=unique_key_predicate,
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()  # Rule 1: If an activity exists, update it.
                .when_not_matched_insert_all()  # Rule 2: If it's a new activity, insert it.
                .execute()
    )
    print("✅ **Successfully saved changes to Delta Lake!**")
except Exception as e:
    print(f"❌ **Error saving to Delta Lake:** {e}")

✅ **Successfully saved changes to Delta Lake!**


#### Para corregir la Fecha final cuando se coloca "00:00:00" en lugar de "23:59:00" al finalizar el día

In [90]:
# Mascara para determinar duracion menor a 0, para volver a calcular las horas. 

mask = ( df['Duracion'] < 0 )

In [ ]:
# Crear una mascara para aplicar los cambios

# Create mask for rows that end with '00:00:00'
mask = df['Duracion'].astype(str).str.endswith('00:00:00') & df['FinEvento'].notna() & ( df['Duracion'] < -1300 )

In [116]:
# Muestra cuantos casos se ha identificado

true_indices = mask[mask].index
len(true_indices.tolist())

154

### Vuelve a calcular la columna 'Duracion' en minutos

In [196]:
# Ejecuta el reemplazo de las horas

#df.loc[mask, 'FinEvento'] = df.loc[mask, 'FinEvento'].astype(str).str[:-8] + '23:59:00'
df['Duracion'] = calcular_minutos_transcurridos(
    df['InicioEvento'],
    df['FinEvento']
)

### Muestra fechas de actividades con posible conflicto

In [5]:
# f'-11 < Duracion < 0'   mayor a -11 y menor a 0
# f"Cuenta == '?' and Alimentador != '·' "
# df.query( f'Fecha.str.contains("2025")', engine='python' ).sort_values(by='Duracion', ascending=False)
# df.query( f"Duracion < 0" ).sort_values(by='id_ot', ascending=False)

df.query( f"Duracion < 0 ", engine='python' ).sort_values(by='Duracion', ascending=False)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
57,11,REDES,Por disposición del ingeniero Ernesto Palacio...,NO PROG,Bomboiza,No,No,No,PREVENTIVO,·,...,2025-09-25 18:09:00,2025-09-25 17:00:00,-69,MORALES RIVERA LUIS ALBERTO,2,No,2-91,"Gualaquiza, El Pangui",161975,OT [02] Alumbrado Zamora 2025-09-25 (012) LM.pdf


In [79]:
# filer DateTime ends with "00:00:00"

filtered_df = df.query('FinEvento.str.endswith("00:00:00")')
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
0,1,informativa,"En apego a la RESOLUCIÓN N° 015-2020-GENERAL, ...",PROG,·,No,No,No,PREVENTIVO,·,...,2020-05-28 00:00:00,2020-05-28 00:00:00,0.0,ALEJANDRO PACHAR AGUSTIN EDUARDO,1,Si,R-120,"YANTZAZA, ZUMBI, CHICAÑA.",47383,OT [04] Cuadrilla Yantzaza 2020-05-28 (022) AA...
90,1,informativa,En apego a la RESOLUCIÓN Nro. 037-2020-GENERAL...,PROG,·,No,No,No,PREVENTIVO,·,...,2020-11-27 00:00:00,2020-11-27 00:00:00,0.0,JARA NARVAEZ GALO SILVERIO,1,Si,4-74,"Zamora, San Marcos",56785,OT [21] Agencia Zamora 2020-11-27 (047) GJ.pdf
230,1,informativa,En apego a la RESOLUCIÓN Nro. 037-2020-GENERAL...,PROG,·,No,No,No,PREVENTIVO,·,...,2021-01-14 00:00:00,2021-01-14 00:00:00,0.0,MORALES RIVERA LUIS ALBERTO,1,Si,R-29,"Zamora, Timbara, Cumbaratza",59327,OT [01] Cuadrilla Zamora 2021-01-14 (012) LM.pdf
248,1,informativa,Se labora en apego a la RESOLUCIÓN N° 012-2021...,PROG,·,No,No,No,PREVENTIVO,·,...,2021-06-21 00:00:00,2021-06-21 00:00:00,0.0,AMARI ORDONEZ JUNIOR IVAN,3,Si,2-102,"El Pangui, San Vicente, El Remolino",67900,OT [08] Cuadrilla El Pangui 2021-06-21 (031) J...
258,1,informativa,"En apego a la RESOLUCIÓN N° 037-2020 GENERAL, ...",PROG,·,No,No,No,·,·,...,2021-03-10 00:00:00,2021-03-10 00:00:00,0.0,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,R-120,Yantzaza,62334,OT [04] Cuadrilla Yantzaza 2021-03-10 (032) AO...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222945,1,informativa,Se coordina los trabajos a realizarse Preparac...,PROG,·,No,No,No,PREVENTIVO,·,...,2024-03-08 00:00:00,2024-03-08 00:00:00,0.0,CHAMBA CANGO PEDRO ROSALINO,1,Si,4-100,"GUAYZIMI, NUEVO PARAISO,NANKAIS.",125798,OT [22] Agencia Yantzaza 2024-03-08 (052) PCH.pdf
243569,1,informativa,"Día festivo en apego a la ley de Feriados ,por...",·,·,No,No,No,·,·,...,2025-02-14 00:00:00,2025-02-14 00:00:00,0.0,CACAY LUZURIAGA ASDRUBAL HUMBERTO,1,Si,R-126,Guayzimi,147711,OT [07] Cuadrilla Guayzimi 2025-02-14 (036) AC...
249348,1,informativa,PLAN DE MANIBRAS EN LA AP LOS ENCUENTROS secto...,·,·,No,No,No,·,·,...,2025-04-26 00:00:00,2025-04-26 00:00:00,0.0,ZHUNAULA GUAMAN VICTOR ANIBAL,1,Si,2-112,Yantzaza- Chicaña,151935,OT [04] Cuadrilla Yantzaza 2025-04-26 (070) VZ...
250413,3,REDES,"Chamico, nos reintegramos con la cuadrilla de ...",PROG,Yantzaza,No,No,No,CORRECTIVO,·,...,2025-04-09 09:10:00,2025-04-09 00:00:00,-550.0,MORALES RIVERA LUIS ALBERTO,2,No,2-91,"Chamico, Benjamin Carrion, Zamora",150852,OT [02] Alumbrado Zamora 2025-04-09 (012) LM.pdf


In [47]:
df.query( f"Duracion < -1300" ).sort_values(by='Duracion', ascending=True)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
1155,18,transporte,Se retorna a Yantzaza,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-11-27 23:40:00,2022-11-27 00:00:00,-1420,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
168298,9,transporte,Nos trasladamos desde el sector de Cuzuntza de...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-03-19 23:39:00,2022-03-19 00:00:00,-1419,MORALES RIVERA LUIS ALBERTO,1,Si,2-110,"El Limón, Cuzuntza.",82958,OT [02] Alumbrado Zamora 2022-03-19 (012) LM.pdf
180003,14,REDES,En el sector de La Y De El Guismi se apoya a ...,NO PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2023-05-28 23:25:00,2023-05-28 00:00:00,-1405,LITUMA CORDOVA CESAR RUBEN,1,Si,R-132,Gualaquiza,108032,OT [09] Cuadrilla Gualaquiza 2023-05-28 (043) ...
135695,7,transporte,Nos trasladamos desde el sector de Guaguayme A...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-12-25 23:23:00,2022-12-25 00:00:00,-1403,MORALES RIVERA LUIS ALBERTO,1,Si,2-110,"Guaguayme Alto, San Carlos de Las Minas.",98957,OT [02] Alumbrado Zamora 2022-12-25 (012) LM.pdf
208445,16,REDES,"Zamora, se coordina con Centro de Maniobras y ...",PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2024-10-10 23:21:00,2024-10-10 00:00:00,-1401,RIOS RIOS FRANCISCO FERNANDO,2,No,2-110,Zamora,139298,OT [01] Cuadrilla Zamora 2024-10-10 (006) FR.pdf
208443,14,REDES,"Zamora, se coordina con Centro de Maniobras y ...",PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2024-10-10 23:21:00,2024-10-10 00:00:00,-1401,RIOS RIOS FRANCISCO FERNANDO,2,No,2-110,Zamora,139298,OT [01] Cuadrilla Zamora 2024-10-10 (006) FR.pdf
36672,17,transporte,Centro de Control Loja Informa que en Sevilla ...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2020-08-07 23:40:00,2020-08-07 00:20:00,-1400,AMARI ORDONEZ JUNIOR IVAN,1,No,R-96,"El Pangui, Gualaquiza",51107,OT [08] Cuadrilla El Pangui 2020-08-07 (031) J...
7170,18,informativa,"DAÑO REPORTADO POR Centro de Control, Desde N...",NO PROG,·,No,No,No,·,·,...,2022-06-15 23:20:00,2022-06-15 00:00:00,-1400,RIOS RIOS FRANCISCO FERNANDO,1,No,2-110,"Zamora, Central Carlos Mora, El Retorno, Río B...",88023,OT [01] Cuadrilla Zamora 2022-06-15 (006) FR.pdf
134687,9,REDES,"Kantzama Bajo, en la estructura. 70787 se cam...",NO PROG,Yacuambi,No,No,No,CORRECTIVO,·,...,2022-04-29 23:20:00,2022-04-29 00:00:00,-1400,SILVA ARMIJOS ROMEL EDUARDO,4,No,2-110,"Guadalupe, Piuntza, El Guayabal, Kantzama Bajo",85366,OT [01] Cuadrilla Zamora 2022-04-29 (007) RS.pdf
142864,19,transporte,Guadalupe - Zamora,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-05-31 23:20:00,2022-05-31 00:00:00,-1400,MACAS CURIPOMA RAMIRO HOMERO,1,No,2-61,"Napintza, San Carlos, Namacuntza",87077,OT [21] Agencia Zamora 2022-05-31 (027) RM.pdf


In [42]:
df.sort_values(by="Duracion",ascending=True).head(10)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
1685,18,transporte,Se retorna a Yantzaza,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-11-27 23:40:00,2022-11-27 00:00:00,-1420,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,2-112,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,OT [04] Cuadrilla Yantzaza 2022-11-27 (032) AO...
167416,9,transporte,Nos trasladamos desde el sector de Cuzuntza de...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-03-19 23:39:00,2022-03-19 00:00:00,-1419,MORALES RIVERA LUIS ALBERTO,1,Si,2-110,"El Limón, Cuzuntza.",82958,OT [02] Alumbrado Zamora 2022-03-19 (012) LM.pdf
152801,19,transporte,Nos trasladamos desde el sector de San Carlos ...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-03-14 23:39:00,2022-03-14 00:00:00,-1419,MORALES RIVERA LUIS ALBERTO,3,No,2-91,Zamora,82559,OT [02] Alumbrado Zamora 2022-03-14 (012) LM.pdf
122481,14,transporte,Nos trasladamos desde el sector La Saquea a la...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2021-01-15 23:35:00,2021-01-15 00:00:00,-1415,SILVA ARMIJOS ROMEL EDUARDO,0,No,,Zamora,59387,OT [01] Cuadrilla Zamora 2021-01-15 (007) RS.pdf
218739,16,transporte,Nos trasladamos de Los Encuentros a la agencia...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2024-04-20 23:35:00,2024-04-20 00:00:00,-1415,ALEJANDRO PACHAR AGUSTIN EDUARDO,1,Si,2-112,"Yantzaza, Zumbi, Chimbutza, Los Encuentros.",128467,OT [04] Cuadrilla Yantzaza 2024-04-20 (022) AA...
128313,13,?,"Los Encuentros. estructura 80238, se revisa e...",NO PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,2021-07-07 23:35:00,2021-07-07 00:00:00,-1415,ALEJANDRO PACHAR AGUSTIN EDUARDO,2,No,4-36,"YANTZAZA, PANGUINTZA, CHICAÑA, KUNKI, LOS ENCU...",69050,OT [04] Cuadrilla Yantzaza 2021-07-07 (022) AA...
45107,7,transporte,"Desde la Estación Científica San Francisco, no...",TRANSP,·,No,No,No,TRANSPORTE,·,...,2019-12-01 23:35:00,2019-12-01 00:00:00,-1415,RIOS RIOS FRANCISCO FERNANDO,1,Si,R-113,"Zamora, San Carlos, Estación Científica San Fr...",39389,OT [01] Cuadrilla Zamora 2019-12-01 (006) FR.pdf
5001,11,transporte,Nos trasladamos de Zumbi a la agencia Yantzaza.,TRANSP,·,No,No,No,TRANSPORTE,·,...,2021-12-26 23:35:00,2021-12-26 00:00:00,-1415,ALEJANDRO PACHAR AGUSTIN EDUARDO,1,Si,2-112,"Yantzaza, El Padmi, Zumbi, Tuntiak.",78392,OT [04] Cuadrilla Yantzaza 2021-12-26 (022) AA...
203632,9,transporte,Nos trasladamos de La Hueca a Yantzaza.,TRANSP,·,No,No,No,TRANSPORTE,·,...,2023-05-19 23:30:00,2023-05-19 00:00:00,-1410,ALEJANDRO PACHAR AGUSTIN EDUARDO,1,No,4-100,"Yantzaza, La Hueca.",107600,OT [04] Cuadrilla Yantzaza 2023-05-19 (022) AA...
166222,2,REDES,Pita. Se recorre y revisa la derivación a la ...,NO PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2022-05-05 23:30:00,2022-05-05 00:00:00,-1410,ALEJANDRO PACHAR AGUSTIN EDUARDO,1,No,2-112,Yantzaza.,85747,OT [04] Cuadrilla Yantzaza 2022-05-05 (022) AA...


In [ ]:
df

# LEGACY

> Analisis posterio entre MongoDB y DeltaLake

## Conectar con DASK Local Cluster

In [11]:
# DASK
from dask.distributed import LocalCluster, as_completed
dask = LocalCluster().get_client()
dask.dashboard_link

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41201 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:45905
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:41201/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39789'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43903'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45879'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39097'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:3979

'http://127.0.0.1:41201/status'

## Recargar Librerias Dinámicamente


In [6]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as ClaseOT
from eerssa import procesarOt as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import generarMatrizActividades as Actividades     # process ot.data["actividades"]
from eerssa import procesarActividades as ActividadesV30

In [13]:
reload( OrdenTrabajo )
reload( Actividades  )
reload( ClaseOT )
reload( ActividadesV30)

<module 'eerssa.procesarActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/procesarActividades.py'>

## Verificacion de OT desde MongoDB hacia DeltaLake

### Descargar y procesar [01] desde MongoDB 

In [ ]:
id_descarga = 97355

try:
  ot_test = CurrentCollection.find_one({'id_ot':id_descarga})
  if not ot_test:
    print(f" [X] No se pudo descargar la OT")
  else:
    activ = ActividadesV30.ConvertirOT_a_ActividadesCSV( ClaseOT.GestionOt.from_v30( ot_test ) )

except Exception as e:
  print(f"{e}")

In [33]:
activ

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
0,1,informativa,Atención de reclamos de Centro de Control (No ...,PROG,·,No,No,No,PREVENTIVO,·,...,2022-11-27 08:00:00,2022-11-27 08:05:00,5,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
1,2,transporte,Se traslada a la parroquia Chicaña en Yantzaza...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-11-27 08:05:00,2022-11-27 08:30:00,25,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
2,3,REDES,Se revisa LM/T en la estructura. 161971 se rep...,NO PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,2022-11-27 08:30:00,2022-11-27 09:50:00,80,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
3,4,transporte,Se traslada a el barrio Jesus del Garan Poder ...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-11-27 09:50:00,2022-11-27 10:10:00,20,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
4,5,REDES,Se revisa LM/T en Yantzaza Av. Ivan Riofrio en...,NO PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2022-11-27 10:10:00,2022-11-27 11:00:00,50,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
5,6,transporte,Se traslada a el barrio el Porvenir en Yantzaz...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-11-27 11:00:00,2022-11-27 11:10:00,10,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
6,7,REDES,Se revisa medidor 1000364290 se encuentra daña...,NO PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2022-11-27 11:10:00,2022-11-27 13:00:00,110,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
7,8,lunch,Se hace uso del Lunch en Yantzaza.,ALIMEN,·,No,No,No,LUNCH,·,...,2022-11-27 13:00:00,2022-11-27 14:00:00,60,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
8,9,REDES,Se continua con los trabajos de corte de veget...,NO PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2022-11-27 14:00:00,2022-11-27 16:20:00,140,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf
9,10,transporte,Se retorna a la Agencia.\rSEGUNDA JORNADA DE T...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2022-11-27 16:20:00,2022-11-27 16:31:00,11,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,·,"Yantzaza, El Zarza, Los Hachos, La Yona",97355,Orden de trabajo Yantzaza 27-11-2022 (AO).pdf


In [35]:
import pyarrow as pa 

edited_data = pa.Table.from_pandas( activ )

try:

    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
                    source=edited_data,
                    predicate=unique_key_predicate,
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()  # Rule 1: If an activity exists, update it.
                .when_not_matched_insert_all()  # Rule 2: If it's a new activity, insert it.
                .when_not_matched_by_source_delete(  # Rule 3: If an old activity is now gone...
                    predicate=f"target.id_ot = {id_descarga}"  # ...delete it, but only for the current OT.
                )
                .execute()
    )
    print("✅ **Successfully saved changes to Delta Lake!**")
except Exception as e:
    print(f"❌ **Error saving to Delta Lake:** {e}")

✅ **Successfully saved changes to Delta Lake!**


### Cursor para obtener todos los "id_ot" desde MongoDB

In [14]:
""" 
   OBTENER TODOS LOS 'id_ot' desde MongoDB
"""

try:
    # 1. Use a projection to only retrieve the 'id_ot' field.
    #    - {'id_ot': 1} means "include this field".
    #    - {'_id': 0} means "exclude the default _id field".
    cursor = CurrentCollection.find({}, {'id_ot': 1, '_id': 0})

    # 2. Create a list from the cursor results using a list comprehension.
    #    This iterates through each document in the cursor and extracts 'id_ot'.
    id_ot_list = [doc['id_ot'] for doc in cursor]

    # 3. Now you have your list of all 'id_ot' values.
    print(f"Successfully retrieved {len(id_ot_list)} 'id_ot' values.")
    if id_ot_list:
        print("First 10 values:", id_ot_list[:10])

except Exception as e:
    print(f"An error occurred: {e}")


An error occurred: name 'CurrentCollection' is not defined


In [15]:
"""
   Obtener todos los 'id_ot' existentes en DeltaLake
"""

delta_ids = df["id_ot"].unique()
len(delta_ids)

2203

In [16]:
"""
   Difentecia de las ot que faltan en DeltaLake
"""
set_mongo = set(id_ot_list)
set_delta = set(delta_ids)

# Find which items in set_delta are not in set_mongo
new_ids_set = set_mongo.difference(set_delta)

# Convert the result back to a list
new_ids_to_process = list(new_ids_set)

print(f"Found {len(new_ids_to_process)} new IDs to be processed.")
# We sort the list here just for a predictable, clean output
print(f"New IDs: {sorted(new_ids_to_process)}")

NameError: name 'id_ot_list' is not defined

In [ ]:
"""
   Descargar y procesar las OT faltantes y añadirlas al Delta Lake
"""
new_data_frames = []
for ot in new_ids_to_process:
  json_ot = CurrentCollection.find_one({"id_ot": ot})
  if not json_ot:
    print(f"No se pudo encontrar la OT con id_ot '{ot}' en MongoDB. Saltando.")
    continue
                
  obj_ot = OrdenTrabajo.GestionOt.from_dict(json_ot)
  new_data_frames.append(Actividades.ConvertirOT_a_ActividadesCSV(obj_ot))

In [ ]:
try:
  new_df = pd.concat(new_data_frames, ignore_index=True)
  write_deltalake(table_path, new_df, mode='append')
  print(f" [ EXITO ] DELTA LAKE Se han añadido {len(new_df)} filas a la tabla Delta en '{table_path}'.")
except Exception as e:
  print(f"Fallo al escribir en la tabla Delta: {e}")

 [ EXITO ] DELTA LAKE Se han añadido 121556 filas a la tabla Delta en './test/deltalake_2025'.


## FULL MONGO DOWNLOAD

Generar un nuevo archivo Delta Lake para unificar versiones - Ejecutado JULIO 2025

### Descarga de OT's desde MongoDB hacia Pickle y Delta Lake 

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

# ... setup client, db, collection
cursor = CurrentCollection.find()
all_documents = cursor.to_list() 
# or simply: all_documents = list(cursor)
print(f"Loaded {len(all_documents)} documents into a list.")
client.close()



Loaded 21879 documents into a list.


In [ ]:
obj_list = []
for document in all_documents:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot) )
df = pd.concat(obj_list, ignore_index=True)


In [ ]:
df["Fecha"] = pd.to_datetime(df["Fecha"])
df["InicioEvento"] = pd.to_datetime(df["InicioEvento"], format='mixed')
df["FinEvento"] = pd.to_datetime(df["FinEvento"], format='mixed')

df.to_pickle("/home/vlad/Documents/mongodb_v23.pkl")

/tmp/ipykernel_316235/3892611823.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["InicioEvento"] = pd.to_datetime(df["InicioEvento"], format='mixed')
/tmp/ipykernel_316235/3892611823.py:3: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["FinEvento"] = pd.to_datetime(df["FinEvento"], format='mixed')


In [ ]:
write_deltalake("/home/vlad/delta_v23", df)

In [ ]:
dt = DeltaTable("/home/vlad/delta_v23")

## Descargar OT faltantes desde MongoDB hacia DeltaLake

## Cargar Pickle para analisis

In [43]:
# DELTA LAKE Connection
# Verify the existence of the DELTA LAKE table
import pandas as pd
import numpy as np
from deltalake import DeltaTable
from datetime import datetime
from pprint import pprint


def borra_time_zone( fecha:str ):
  """
  Esta función elimina el componenete de Time Zone y deja solamente la fecha y hora. 
  En caso de que no contenga este componente deja el String intacto. 
  """
  fecha_inicio = fecha.replace('T', ' ').split()
  fecha_inicio = fecha_inicio[0]+' '+fecha_inicio[-1]
  return fecha_inicio

if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {table_path}")

df.info()

Conectado a la tabla Delta Lake en: /home/vlad/delta_V30
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31308 entries, 0 to 31307
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           31308 non-null  int64 
 1   Cuenta         31308 non-null  object
 2   Evento         31308 non-null  object
 3   Actividad      31308 non-null  object
 4   Alimentador    31308 non-null  object
 5   Primario       31308 non-null  object
 6   Desconexion    31308 non-null  object
 7   SIG            31308 non-null  object
 8   Tipo           31308 non-null  object
 9   Materiales     31308 non-null  object
 10  Cuadrilla      31308 non-null  object
 11  Dia            31308 non-null  object
 12  Fecha          31308 non-null  object
 13  InicioEvento   31308 non-null  object
 14  FinEvento      31308 non-null  object
 15  Duracion       31308 non-null  int64 
 16  Responsable    31308 non-null  object
 17  Colaboradore

In [31]:
variant = df.copy()

In [44]:
df['InicioEvento'] = df['InicioEvento'].apply( lambda x: borra_time_zone(x))
df['FinEvento'] = df['FinEvento'].apply( lambda x: borra_time_zone(x))

In [34]:
dt.version()

147

In [ ]:
# Perform the merge operation
print("\n--- Merging changes back into Delta Table ---")

(
    dt.merge(
        source=df,
        predicate="target.id = source.id",
        source_alias="source",
        target_alias="target"
    )
    .when_matched_update_all()  # If id matches, update the row
    .when_not_matched_insert_all()  # If a new id is in the source, insert it
    .execute()
)

print("Merge complete.")

### ¿Son todas los items en 'Fecha' validos?

In [45]:
def is_valid_utc_format(text_input):
    """
    Checks if a string can be converted to a timezone-aware datetime.

    The function tests against a specific ISO 8601 format that includes a
    UTC offset, like "2024-05-31T00:00:00-05:00".

    Args:
        text_input: The string or value to check.

    Returns:
        - True: if the input is a string and matches the format.
        - False: if the input is not a string or does not match the format.
        - pd.NaT: if the input is a null-like value (e.g., None, np.nan).
    """
    # 1. Handle null-like inputs first
    if pd.isna(text_input):
        return pd.NaT

    # 2. Ensure the input is a string before attempting to parse
    if not isinstance(text_input, str):
        return False

    # 3. Try to parse the string using the specific format
    try:
        # The format string matches the user's example.
        # %Y: 4-digit year
        # %m: 2-digit month
        # %d: 2-digit day
        # T: Literal 'T' separator
        # %H:%M:%S: Hour, minute, second
        # %z: UTC offset (e.g., -0500). Pandas extends this to handle
        #     the colon format (-05:00) as well.
        # errors='raise' ensures that any parsing failure raises an exception.
        pd.to_datetime(text_input, format="%Y-%m-%d %H:%M:%S", errors='raise')
        return True
    except ValueError:
        # This exception is raised if the string does not match the format.
        return False



In [47]:
df['InicioEvento'][0]

'2024-07-17 08:00:00'

In [37]:
fechas_validas = variant['InicioEvento'].apply( lambda x: is_valid_utc_format(x) )
fechas_validas.unique()

array([ True, False])

In [39]:
false_indices = np.where(~fechas_validas)[0]
len(false_indices)

18151

In [41]:
last_wrong_date = false_indices[-1]

In [ ]:
# 2023-02-20T00:00:00-05:00 00:00:01

In [42]:
variant.iloc[last_wrong_date]

Item                                                             4
Cuenta                                                   se_labora
Evento           SE LABORA: RM, RY\nNo se presentan novedades e...
Actividad                                                   LABORA
Alimentador                                                      ·
Primario                                                        No
Desconexion                                                     No
SIG                                                             No
Tipo                                                             ·
Materiales                                                       ·
Cuadrilla                                         Zamora (Agencia)
Dia                                                        viernes
Fecha                                    2022-02-11T00:00:00-05:00
InicioEvento                    2022-02-11T00:00:00-05:00 00:00:01
FinEvento                       2022-02-11T00:00:00-05:00 00:0

In [ ]:
import datetime
# 2. Define a function to safely get the date
def safe_to_date(value):
    # Check if the value is a Timestamp or datetime object
    if isinstance(value, (pd.Timestamp, datetime.datetime)):
        return value.date()
    # If it's already a date object, just return it
    elif isinstance(value, datetime.date):
        return value
    # For any other type, return NaT (Not a Time)
    else:
        return pd.NaT

In [ ]:
df['dates_equal_Inicio'] = (df['Fecha'].dt.date == df['InicioEvento'].apply(safe_to_date))
df['dates_equal_Fin'] = (df['Fecha'].dt.date == df['FinEvento'].apply(safe_to_date))

In [ ]:
df['dates_equal_Inicio'].unique()

array([ True, False])

In [ ]:
falla_inicio = df.query("dates_equal_Inicio == False")
#falla_inicio[["Item","Responsable","id_ot","Fecha","InicioEvento","FinEvento"]]
falla_inicio["id_ot"].unique()

array([155505, 155762, 145572, 144647, 148359, 147423, 148549, 148277,
       148246, 146584, 147455, 147804, 148333, 149748, 149726, 149242,
       150570, 137497, 139625, 138768, 142840, 139823, 139690, 134776,
       137787, 128220, 139615, 144309, 141353, 144417, 134989, 143002,
       140128, 134479, 137253, 125688, 110252, 105181, 114346, 119028,
       116773, 120598, 114096, 102758, 106048, 115879, 113463, 100339,
       104339, 111938, 120285, 114645, 109933, 111049,  84875,  89820,
        85621,  96068,  80898,  83812,  82240,  92859,  81700,  86257,
        83222, 156391,  75212,  67725,  76277,  73194,  70460,  76774,
        67486,  60815,  64571,  54263,  50513])

## VERIFICACIÓN de OTs en Mongo DB

1. Se extrae el listado de todos los `id_ot` de los PDF existentes
2. Se verifica este listado con los documentos en `MongoDB`
3. Se verifica este listado con los documentos en `DeltaLake`

### Conexion con DASK

In [18]:
# 1. Listado de OTs con sus ID:

# Directorio Raiz de las OT (año) para validar

#root_dir = ("/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/"
#            +
#            "2020")

root_dir = "/home/vlad/Documents/000 OTs Antiguas/2020"

list_pdfs = []
for path in Path( root_dir ).glob("**/*.pdf"):
    list_pdfs.append( str(path) )
    list_pdfs.sort()

print(f" Se han encontrado un total de: {len(list_pdfs)} Ordenes de Trabajo" )

start_time = time.time()
start_datetime = datetime.now()
print( f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n" )

# Helper function to call the method on the result of a future
def call_load_ot(orden_trabajo_object):
    """
    Takes the result of the first task (an OrdenTrabajo object) 
    and calls the load_ot() method on it.
    """
    return orden_trabajo_object.load_ot()

# 1. Submit the first batch of tasks
# This returns a list of futures, same as before.
futures_step1 = [dask.submit(OrdenTrabajo.GestionOt, file) for file in list_pdfs]

# 2. Submit the second batch of tasks, feeding the first futures as input
futures_step2 = [dask.submit(call_load_ot, f) for f in futures_step1]

# 3. Now, gather only the FINAL results
# This single call executes the entire graph (both GestionOt and load_ot) in parallel.
obj_lists_dask = dask.gather(futures_step2)


end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")




 Se han encontrado un total de: 3054 Ordenes de Trabajo
Hora de inicio: 2025-07-24 10:21:50




   Procesados todos los 3054 items. Tiempo transcurrido: 294.74 segundos.
   Hora Final : 2025-07-24 10:26:45


### Examinado los objetos en `futures_step2`

In [20]:
dask.cancel([futures_step1,futures_step2])

### Buscamos docuementos faltantes en Mongo DB

In [19]:
# 2. Obtenemos los id_OT para compararlos con la base de datos en MongoDB

# Step 1: Collect all the IDs from your local list into a new list.
all_ids = [ot.id_ot for ot in obj_lists_dask]

# Step 2: Use the "$in" operator to find all documents in the database
# that match any ID in your list. This is ONE efficient query.
existing_docs_cursor = CurrentCollection.find(
    {"id_ot": {"$in": all_ids}},
    {"id_ot": 1}  # Projection: only return the _id and id_ot fields for efficiency
)

# Step 3: Create a set of the IDs that were actually found in the database.
# Sets provide very fast lookups.
ids_in_db = {doc['id_ot'] for doc in existing_docs_cursor}

# Step 4: Find the difference between the set of all IDs and the set of IDs found in the DB.
missing_ot_ids = set(all_ids) - ids_in_db

# Print the missing IDs
for current_id in missing_ot_ids:
    print(f" [X] The ot with id: {current_id} is not in the Database ")

# Your final result is a list of the missing IDs
missing_ot = list(missing_ot_ids)


 [X] The ot with id: 52335 is not in the Database 


In [31]:
del obj_lists_dask

#### Existen OTs en estado PDF repetidas en las carpetas

Esta es la razon de que no coincidan los numeros

In [17]:
# Buscando en MongoDB con REGEX:

csv = "/home/vlad/Documents/temp_borrar/a-reporete_del_reporte/2024-reportes/ots_query_mongo_2-24/eerssa.ot_v22 OTS del 2024.csv"

#Load file into Pandas
csv_df = pd.read_csv( csv )
csv_df.head()

regex_id = csv_df["id_ot"].tolist()
len(regex_id)

4293

In [24]:
ids_not_pdf = set(all_ids) - set(regex_id)
len(ids_not_pdf)

3

In [28]:
print(f" Total Elementos en all_ids : {len(all_ids)} Elementos unicos : {len(set(all_ids))}")

 Total Elementos en all_ids : 4316 Elementos unicos : 4296


In [30]:
from collections import Counter

pdf_contador  = Counter(all_ids)

repeated_items = {item: count for item, count in pdf_contador.items() if count > 1 }
print("\nRepeated items and their counts:")
pprint(repeated_items)


Repeated items and their counts:
{122020: 3,
 124079: 2,
 125318: 2,
 125661: 2,
 125698: 2,
 125755: 2,
 125796: 2,
 125860: 2,
 125945: 2,
 126011: 2,
 126083: 2,
 129097: 2,
 129152: 2,
 129161: 2,
 129167: 2,
 129927: 2,
 131619: 3,
 134113: 2}


### NUEVO DELTA LAKE Descargar todo el 2025

Para iniciar crearemos un DeltaLake de las OTs del 2025

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

regex_pattern = re.compile("^2025")
query_filter = {"fecha": regex_pattern}

# Use find() to get a cursor that points to all matching documents
cursor = CurrentCollection.find(query_filter)

obj_list = []
for document in cursor:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot))
  #print(f"Processing document with id_ot: {document.get('id_ot')}")
df = pd.concat(obj_list, ignore_index=True)


In [3]:
from deltalake import DeltaTable, write_deltalake
DELTA_TABLE_PATH_ON_HOST = "/home/vlad/delta_V30"
# First message written with
#write_deltalake(DELTA_TABLE_PATH_ON_HOST, df)

In [5]:
dt = DeltaTable( DELTA_TABLE_PATH_ON_HOST)
dt.version()

1

In [6]:
dt.files()

/tmp/ipykernel_142148/4072315134.py:1: DeprecationWarning: Call to deprecated method files. (Not compatible with modern delta features (e.g. shallow clones). Use `file_uris` instead.) -- Deprecated since version 1.0.0.
  dt.files()


['part-00001-a3f8be11-e02f-420c-acec-5a68a5639cce-c000.snappy.parquet',
 'part-00001-bc789fbb-734a-495b-94ed-e2665afe366c-c000.snappy.parquet']

In [33]:
df = dt.to_pandas()

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61960 entries, 0 to 61959
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           61960 non-null  int64 
 1   Cuenta         61960 non-null  object
 2   Evento         61960 non-null  object
 3   Actividad      61813 non-null  object
 4   Alimentador    20805 non-null  object
 5   Primario       61960 non-null  object
 6   Desconexion    61960 non-null  object
 7   SIG            61960 non-null  object
 8   Tipo           53789 non-null  object
 9   Materiales     61960 non-null  object
 10  Cuadrilla      61960 non-null  object
 11  Dia            61960 non-null  object
 12  Fecha          61960 non-null  object
 13  InicioEvento   61960 non-null  object
 14  FinEvento      61958 non-null  object
 15  Duracion       61960 non-null  int64 
 16  Responsable    61960 non-null  object
 17  Colaboradores  61960 non-null  int64 
 18  HorasExtra     61960 non-n